In [ ]:
# Load data from database
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

# Database connection details
load_dotenv()

DB_USER = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')
DB_HOST = os.getenv('DB_HOST')
DB_PORT = os.getenv('DB_PORT')
DB_NAME = os.getenv('DB_NAME')

# Dataset folder path
DATASET_PATH = os.getenv('DATASET_PATH')

# Create the connection engine
engine = create_engine(
    f'postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}'
)

# Load the whole table
df = pd.read_sql("SELECT * FROM email_detection_dataset", engine)
df = df[["content", "class"]]

df.head()


,content,class
0,Supply Quality China's EXCLUSIVE dimensions at...,1
1,over. SidLet me know. Thx.,0
2,"Dear Friend,Greetings to you.I wish to accost ...",1
3,MR. CHEUNG PUIHANG SENG BANK LTD.DES VOEUX RD....,1
4,Not a surprising assessment from Embassy.,0


In [2]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import re

#data cleaning
# Drop rows with missing text
df = df.dropna(subset=['content']).copy()

#Clean the text column
def clean_text(text):
    text = text.lower()
    text = re.sub(r'https?\S+', '', text)  # Remove links
    text = re.sub(r'[^a-z\s]', '', text)   # Remove punctuation/numbers
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['content'] = df['content'].apply(clean_text)
y = df['class']

In [3]:
# Vectorize text
vectorizer = CountVectorizer(max_features=1000)
X = vectorizer.fit_transform(df['content'])

X_df = pd.DataFrame(X.toarray(), columns=vectorizer.get_feature_names_out())

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [4]:
from sklearn.metrics import classification_report

# Neural Network
nn_model = MLPClassifier(hidden_layer_sizes=(100,), max_iter=300, random_state=42)
nn_model.fit(X_train, y_train)
nn_acc = accuracy_score(y_test, nn_model.predict(X_test))
print("Accuracy Score for Neural Network:", round(nn_acc, 4))
print(classification_report(y_test, nn_model.predict(X_test), target_names=['Non-Fraud', 'Fraud']))

# Confusion matrix
nn_cm = confusion_matrix(y_test, nn_model.predict(X_test))
disp = ConfusionMatrixDisplay(confusion_matrix=nn_cm, display_labels=['Non-Fraud', 'Fraud'])

Accuracy Score for Neural Network: 0.9711
              precision    recall  f1-score   support

   Non-Fraud       0.96      0.99      0.97      1341
       Fraud       0.99      0.94      0.97      1045

    accuracy                           0.97      2386
   macro avg       0.97      0.97      0.97      2386
weighted avg       0.97      0.97      0.97      2386



In [5]:
import joblib

# Save the model
joblib.dump(nn_model, "email_detection_nn_model.pkl")

['email_detection_nn_model.pkl']